In [ ]:
%%capture
!pip install --upgrade pip
!pip install datasets[audio]
!pip install -q --upgrade transformers huggingface_hub
!pip install -q -U torchao
!pip install datacollective
!pip install pylangacq --break-system-packages
!pip install praatio
!pip install pympi-ling
!pip install pyDataverse

#### YECS

In [ ]:
import ast # Import ast for literal_eval
import os
import re
import json
import unicodedata
import tarfile
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch, torchaudio
import pylangacq
import pympi
import seaborn as sns
import tempfile
import itertools
import librosa
import soundfile as sf
import wave
import zipfile



from IPython.display import Audio as AudDisp, display
from collections import defaultdict, Counter
from huggingface_hub import notebook_login,login, snapshot_download
from datacollective import download_dataset
from functools import lru_cache
from pathlib import Path
from rustling.chat import CHAT  # pylangacq now depends on this directly
from pydub import AudioSegment
from datasets import Dataset, DatasetDict, Audio, load_dataset, Features, Value, List, concatenate_datasets
from tqdm.notebook import tqdm

In [ ]:
os.environ["MDC_API_KEY"] = "<mdc-api-key>"
login("<hf-api-key>")
lang_code = "yor_ng"
yor = download_dataset("cmo09pqp300gbnx07xcl42los")

In [ ]:
extract_path = "./yor_ng"
with tarfile.open(yor, "r:gz") as tar:
    tar.extractall(extract_path)

for root, dirs, files in os.walk(extract_path):
    print(root, len(files), "files")
    if files:
        print("  sample:", files[:5])

/tmp/ipykernel_2559/3540760994.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_path)


./yor_ng 0 files
./yor_ng/YECS_Corpus 0 files
./yor_ng/YECS_Corpus/metadata 3 files
  sample: ['test_metadata.csv', 'val_metadata.csv', 'train_metadata.csv']
./yor_ng/YECS_Corpus/audio 0 files
./yor_ng/YECS_Corpus/audio/test 9949 files
  sample: ['AUD_092553.wav', 'AUD_082541.wav', 'AUD_017463.wav', 'AUD_007951.wav', 'AUD_020415.wav']
./yor_ng/YECS_Corpus/audio/train 80015 files
  sample: ['AUD_091876.wav', 'AUD_078310.wav', 'AUD_043521.wav', 'AUD_059319.wav', 'AUD_007151.wav']
./yor_ng/YECS_Corpus/audio/val 9966 files
  sample: ['AUD_083765.wav', 'AUD_020582.wav', 'AUD_096786.wav', 'AUD_071991.wav', 'AUD_018365.wav']


In [ ]:
train_data = pd.read_csv("yor_ng/YECS_Corpus/metadata/train_metadata.csv")

In [ ]:
train_data.head()

,audio_id,speaker_id,prompt_id,split,file_name,prompt,duration,language_tags,emotions,domain,gender
0,AUD_000001,SPK_0097,PRT_0001,train,AUD_000001.wav,Ewé to green gan ma wà healthy for plant. .,4.98,"[{'language': 'yo', 'word': 'Ewé'}, {'language...",neutral,Science,Male
1,AUD_000002,SPK_0097,PRT_0002,train,AUD_000002.wav,"Ẹ jọ̀ọ́, má ṣe fi pipette tip sí. reuse.",4.14,"[{'language': 'yo', 'word': 'Ẹ'}, {'language':...",neutral,Science,Male
2,AUD_000004,SPK_0109,PRT_0004,train,AUD_000004.wav,"When hunger is taken out of poverty, abuse bùṣe",4.62,"[{'language': 'en', 'word': 'When'}, {'languag...",neutral,General,Female
3,AUD_000005,SPK_0109,PRT_0005,train,AUD_000005.wav,Egungun ẹlẹru helps to cleanse the village by ...,4.80,"[{'language': 'yo', 'word': 'Egungun'}, {'lang...",neutral,General,Female
4,AUD_000006,SPK_0109,PRT_0006,train,AUD_000006.wav,Ọlámidé is an oní ìwà tútù bi dove,4.50,"[{'language': 'yo', 'word': 'Ọlámidé'}, {'la...",neutral,General,Female


In [ ]:
hf_features = Features({
    "filename": Value("string"),
    "audio": Audio(sampling_rate=16000),
    "speaker": Value("string"),
    "transcript": Value("string"),
    "language_id_per_token": List(Value("string")),
    "language": Value("string"),
    "duration_sec": Value("float64"),
})

In [ ]:
def get_audio_duration(file_path):
    info = sf.info(file_path)
    return info.frames / float(info.samplerate)

def prepare_audio(file_path):
    data, sample_rate = librosa.load(file_path, sr=16000)
    sf.write(file_path, data, sample_rate)

def get_lid_tokens(lang_list):
    try:
        # Use ast.literal_eval for robust parsing of Python list of dicts string
        parsed_list = ast.literal_eval(lang_list)
    except (ValueError, SyntaxError) as e:
        # Fallback in case of parsing errors, though ast.literal_eval is generally robust for this format
        raise ValueError(f"Error parsing language tags: {e} for input: {lang_list}")

    langs = []
    for lid in parsed_list:
        if lid["language"].lower() == 'yo':
            langs.append('yor')
        elif lid["language"].lower() == 'en':
            langs.append('eng')
        else:
            raise ValueError(f"Unknown language code: {lid['language']}") # More specific error message
    return langs

def curate_data(row, audio_dir, lang_extractor, duration_func, audio_func, lang_code):
    audio_path = Path(audio_dir) / row["file_name"]
    audio_func(audio_path)
    speaker = row["speaker_id"]
    lang_per_token = []
    if row.get("language_tags", None):
        lang_per_token = get_lid_tokens(row["language_tags"])
    if not isinstance(speaker, str):
        speaker = ""
    return {
        "filename": audio_path.stem,
        "audio": str(audio_path),
        "speaker": speaker,
        "transcript": row.get("prompt", "") ,
        "language_id_per_token": lang_per_token,
        "lang": lang_code,
        "duration_sec": row["duration"],
    }

In [ ]:
YECS_CORPUS = "yor_ng/YECS_Corpus/"
train_data = pd.read_csv("yor_ng/YECS_Corpus/metadata/train_metadata.csv")
val_data = pd.read_csv("yor_ng/YECS_Corpus/metadata/val_metadata.csv")
test_data = pd.read_csv("yor_ng/YECS_Corpus/metadata/test_metadata.csv")

In [ ]:
def process(df, split, audio_key, audio_dir, log_file, lang_code):
    rows = []
    log_file = f"{log_file}.log"
    processed_log = open(log_file, "a")
    for id, row in tqdm(df.iterrows()):
        if row[audio_key] in {l.strip() for l in open(log_file)} if Path(log_file).exists() else False:
            continue
        try:
            processed = curate_data(row, audio_dir, get_lid_tokens, get_audio_duration, prepare_audio, lang_code)
        except Exception as e:
            print(f"failed row {id} ({row[audio_key]}): {e}")
            continue
        rows.append(processed)
        processed_log.write(row[audio_key] + "\n")
        processed_log.flush()
    processed_log.close()
    return rows

In [ ]:
train_rows = process(train_data, "train", "file_name", YECS_CORPUS + "audio/train", "train_log", lang_code)

0it [00:00, ?it/s]

In [ ]:
val_rows = process(val_data, "val", "file_name", YECS_CORPUS + "audio/val", "val_log", lang_code)

In [ ]:
test_data.columns

Index(['audio_id', 'speaker_id', 'split', 'file_name', 'duration', 'gender'], dtype='object')

In [ ]:
test_rows = process(test_data, "test", "file_name", YECS_CORPUS + "audio/test", "test_log", lang_code)

0it [00:00, ?it/s]

In [ ]:
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_rows).rename_columns({"lang" : "language"}).cast(hf_features),
    "val" : Dataset.from_list(val_rows).rename_columns({"lang" : "language"}).cast(hf_features),
    "test": Dataset.from_list(test_rows).rename_columns({"lang" : "language"}).cast(hf_features),
})

Casting the dataset:   0%|          | 0/80015 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9966 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9949 [00:00<?, ? examples/s]

In [ ]:
display(AudDisp(dataset_dict["train"][3590]["audio"]["array"], rate=16_000))

In [ ]:
dataset_dict.push_to_hub("nolimitsxl/yecs_lyngual_labs")

Uploading the dataset shards:   0%|          | 0/23 [00:00<?, ? shards/s]

Map:   0%|          | 0/3479 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpng42ijdx.parquet    :   0%|          |  638kB /  489MB            

CommitInfo(commit_url='https://huggingface.co/datasets/nolimitsxl/yecs_lyngual_labs/commit/62deef787c6e0caf52a50ca37e28da9430573487', commit_message='Upload dataset', commit_description='', oid='62deef787c6e0caf52a50ca37e28da9430573487', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nolimitsxl/yecs_lyngual_labs', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nolimitsxl/yecs_lyngual_labs'), pr_revision=None, pr_num=None)

### OpenSLR Data

In [ ]:
%%capture
!wget -P /content/open_slr/70 -r -np -nd -A "*.tsv,*.zip,*.html,LICENSE" https://www.openslr.org/resources/70/
!wget -P /content/open_slr/86 -r -np -nd -A "*.tsv,*.zip,*.html,LICENSE" https://www.openslr.org/resources/86/

In [ ]:
def extract_zip_file(zip_file_path, extract_dir):
    """
    Extracts the contents of a zip file to a specified directory.

    Args:
        zip_file_path (str): The path to the zip file.
        extract_dir (str): The directory where the contents should be extracted.
    """
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir)

    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Successfully extracted '{zip_file_path}' to '{extract_dir}'")
    except zipfile.BadZipFile:
        print(f"Error: '{zip_file_path}' is not a valid zip file or is corrupted.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


def get_audio_duration(file_path):
    info = sf.info(file_path)
    return info.frames / float(info.samplerate)

def prepare_audio(file_path):
    data, sample_rate = librosa.load(file_path, sr=16000)
    sf.write(file_path, data, sample_rate)

def get_lid_tokens(transcript, lang):
    tokens = transcript.split()
    langs = []
    for token in tokens:
        if not (token.startswith("[") and token.endswith("]")):
            langs.append(lang)
    return langs

def curate_data(row, audio_dir, lang_extractor, duration_func, audio_func, lang_code):
    audio_path = Path(audio_dir) / f"{row["filename"]}.wav"
    audio_func(audio_path)
    duration = get_audio_duration(audio_path)
    speaker = "male" if row["filename"].split("_")[0][-1] == "m" else "female"
    lang_per_token = get_lid_tokens(row["transcript"], "yor")
    return {
        "filename": audio_path.stem,
        "audio": str(audio_path),
        "speaker": speaker,
        "transcript": row.get("transcript", "") ,
        "language_id_per_token": lang_per_token,
        "language": lang_code,
        "duration_sec": duration,
    }

def process(df, split, audio_key, audio_dir, log_file, lang_code):
    rows = []
    log_file = f"{log_file}.log"
    processed_log = open(log_file, "a")
    for id, row in tqdm(df.iterrows()):
        if row[audio_key] in {l.strip() for l in open(log_file)} if Path(log_file).exists() else False:
            continue
        try:
            processed = curate_data(row, audio_dir, get_lid_tokens, get_audio_duration, prepare_audio, lang_code)
        except Exception as e:
            print(f"failed row {id} ({row[audio_key]}): {e}")
            continue
        rows.append(processed)
        processed_log.write(row[audio_key] + "\n")
        processed_log.flush()
    processed_log.close()
    return rows

#### Yoruba

In [ ]:
DIR = "/content/open_slr/86/"
extract_zip_file(DIR + "yo_ng_male.zip", DIR + "extracted_yo_ng_male")
extract_zip_file(DIR + "yo_ng_female.zip", DIR + "extracted_yo_ng_female")

Successfully extracted '/content/open_slr/86/yo_ng_male.zip' to '/content/open_slr/86/extracted_yo_ng_male'
Successfully extracted '/content/open_slr/86/yo_ng_female.zip' to '/content/open_slr/86/extracted_yo_ng_female'


In [ ]:
male_tsv = pd.read_csv(DIR + "line_index_male.tsv", sep="\t", header=None, names=["filename", "transcript"])
female_tsv = pd.read_csv(DIR + "line_index_female.tsv", sep="\t", header=None, names=["filename", "transcript"])

In [ ]:
male_rows = process(male_tsv, "all", "filename", DIR  + "extracted_yo_ng_male", "train_log", "yor_ng")
female_rows = process(female_tsv, "all", "filename", DIR  + "extracted_yo_ng_female", "train_log", "yor_ng")
male_rows = Dataset.from_list(male_rows).cast(hf_features)
female_rows = Dataset.from_list(female_rows).cast(hf_features)

0it [00:00, ?it/s]

In [ ]:
AudDisp(female_rows[39]["audio"]["array"], rate=16000)

In [ ]:
all_rows = concatenate_datasets([male_rows, female_rows])
all_rows = all_rows.train_test_split(test_size=0.2,seed=42)
all_rows.push_to_hub("nolimitsxl/open_slr_lang_resource", config_name="yor_ng")

#### English (Ng)

In [ ]:
DIR = "/content/open_slr/70/"
extract_zip_file(DIR + "en_ng_male.zip", DIR + "extracted_en_ng_male")
extract_zip_file(DIR + "en_ng_female.zip", DIR + "extracted_en_ng_female")

Successfully extracted '/content/open_slr/70/en_ng_male.zip' to '/content/open_slr/70/extracted_en_ng_male'
Successfully extracted '/content/open_slr/70/en_ng_female.zip' to '/content/open_slr/70/extracted_en_ng_female'


In [ ]:
male_tsv = pd.read_csv(DIR + "line_index_male.tsv", sep="\t", header=None, names=["filename", "transcript"])
female_tsv = pd.read_csv(DIR + "line_index_female.tsv", sep="\t", header=None, names=["filename", "transcript"])

In [ ]:
male_rows = process(male_tsv, "all", "filename", DIR  + "extracted_en_ng_male", "train_log", "eng_ng")
female_rows = process(female_tsv, "all", "filename", DIR  + "extracted_en_ng_female", "train_log", "eng_ng")
male_rows = Dataset.from_list(male_rows).cast(hf_features)
female_rows = Dataset.from_list(female_rows).cast(hf_features)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Casting the dataset:   0%|          | 0/1314 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2045 [00:00<?, ? examples/s]

In [ ]:
AudDisp(male_rows[39]["audio"]["array"], rate=16000)

In [ ]:
all_rows = concatenate_datasets([male_rows, female_rows])
all_rows = all_rows.train_test_split(test_size=0.2,seed=42)
all_rows.push_to_hub("nolimitsxl/open_slr_lang_resource", config_name="eng_ng")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/1344 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp7lw0g0s4.parquet    :   2%|1         | 2.41MB /  133MB            

CommitInfo(commit_url='https://huggingface.co/datasets/nolimitsxl/open_slr_lang_resource/commit/336d0247ca29276db0ea4f4888672563b2f4406e', commit_message='Upload dataset', commit_description='', oid='336d0247ca29276db0ea4f4888672563b2f4406e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nolimitsxl/open_slr_lang_resource', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nolimitsxl/open_slr_lang_resource'), pr_revision=None, pr_num=None)

In [ ]:
!rm -rf /content/open_slr
!rm -rf /content/yor_ng

### YFACC (Yoruba)

In [ ]:
!wget -O /content/yfacc_v6.tar.gz "https://www.dropbox.com/scl/fi/6x17dliu9icp4n6g5mrbw/yfacc_v6.tar.gz?rlkey=b3y46kl5bx2k0xs1ljwxe6pty&e=1&dl=1"

--2026-09-17 06:59:31--  https://www.dropbox.com/scl/fi/6x17dliu9icp4n6g5mrbw/yfacc_v6.tar.gz?rlkey=b3y46kl5bx2k0xs1ljwxe6pty&e=1&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.80.18, 2620:100:6057:18::a27d:d12
Connecting to www.dropbox.com (www.dropbox.com)|162.125.80.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc708e32c218bbac8e950fb66058.dl.dropboxusercontent.com/cd/0/inline/DIRFYPcJDbig8oIPqsUkMXNedpTaN-1sJnE44jpMLJMfHNcS8GzOL9HnzUffCxecoLjh4VJiyOX62oFfxPFO8Hm2dTMd-7NBJ0ArnKt29VQ1z_BNsMt94YULCSYfVh4VMJQ1827GUsTH_Rk9A7xWIiVv/file?dl=1# [following]
--2026-09-17 06:59:32--  https://uc708e32c218bbac8e950fb66058.dl.dropboxusercontent.com/cd/0/inline/DIRFYPcJDbig8oIPqsUkMXNedpTaN-1sJnE44jpMLJMfHNcS8GzOL9HnzUffCxecoLjh4VJiyOX62oFfxPFO8Hm2dTMd-7NBJ0ArnKt29VQ1z_BNsMt94YULCSYfVh4VMJQ1827GUsTH_Rk9A7xWIiVv/file?dl=1
Resolving uc708e32c218bbac8e950fb66058.dl.dropboxusercontent.com (uc708e32c218bbac8e950fb66058.dl.dropboxusercontent.com)

In [ ]:
extract_path = "./yor_ng"
yor = "/content/yfacc_v6.tar.gz"
with tarfile.open(yor, "r:gz") as tar:
    tar.extractall(extract_path)

for root, dirs, files in os.walk(extract_path):
    print(root, len(files), "files")
    if files:
        print("  sample:", files[:5])

/tmp/ipykernel_2559/502936351.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_path)


./yor_ng 0 files
./yor_ng/yfacc_v6 1 files
  sample: ['readme.md']
./yor_ng/yfacc_v6/flickr_audio_yoruba_test 519 files
  sample: ['S001_2317714088_bcd081f926_0.wav', 'S001_308487515_7852928f90_0.wav', 'S001_2083434441_a93bc6306b_0.wav', 'S001_3208074567_ac44aeb3f3_0.wav', 'S001_2216695423_1362cb25f3_0.wav']
./yor_ng/yfacc_v6/flickr_audio_yoruba_dev 514 files
  sample: ['S001_146098876_0d99d7fb98_0.wav', 'S001_2913207978_9e9624e249_0.wav', 'S001_2826574228_c63009e473_0.wav', 'S001_2201192417_d934730fea_0.wav', 'S001_2406591500_403f145905_0.wav']
./yor_ng/yfacc_v6/Flickr8k_text 5 files
  sample: ['keywords.8_yoruba.txt', 'Flickr8k.token.test_yoruba.txt', 'eng_yoruba_keywords.txt', 'Flickr8k.token.dev_yoruba.txt', 'Flickr8k.token.train_yoruba.txt']
./yor_ng/yfacc_v6/flickr_audio_yoruba_train 5001 files
  sample: ['S001_3606084228_6286a52875_0.wav', 'S001_1089181217_ee1167f7af_0.wav', 'S001_1463732807_0cdf4f22c7_0.wav', 'S001_2755952680_68a0a1fa42_0.wav', 'S001_543326592_70bd4d8602_0.wav'

In [ ]:
from praatio import textgrid

tg = textgrid.openTextgrid("/content/yor_ng/yfacc_v6/Flickr8k_alignment/S001_1012212859_01547e3f17_0.TextGrid", includeEmptyIntervals=False)

for tier_name in tg.tierNames:
    tier = tg.getTier(tier_name)
    for entry in tier.entries:
        start, end, label = entry
        print(tier_name, start, end, label)

FileNotFoundError: [Errno 2] No such file or directory: '/content/yor_ng/yfacc_v6/Flickr8k_alignment/S001_1012212859_01547e3f17_0.TextGrid'

In [ ]:
"0" * (10 - len("10122128"))

'00'

In [ ]:
DIR = "/content/yor_ng/yfacc_v6/"
def format_to_df(fp):
    texts = Path(fp).read_text()
    rows = texts.splitlines()
    frows = []
    for row in tqdm(rows):
        splits = row.split("\t")
        splits = [sp for sp in splits if sp.strip() != '']
        filename, text = splits[0], splits[1]
        filename = filename.rstrip().rstrip(".jpg#0").split("_")
        filename[1] = filename[1][:10] + "0" * (10 - len(filename[1]))
        filename = "S001_" + "_".join(filename) + "_0.wav"
        frows.append([filename, text])
    return pd.DataFrame(frows, columns=["filename", "transcript"])


train = format_to_df(DIR + "Flickr8k_text/Flickr8k.token.train_yoruba.txt")
test = format_to_df(DIR + "Flickr8k_text/Flickr8k.token.test_yoruba.txt")
dev = format_to_df(DIR + "Flickr8k_text/Flickr8k.token.dev_yoruba.txt")

  0%|          | 0/5207 [00:00<?, ?it/s]

  0%|          | 0/526 [00:00<?, ?it/s]

  0%|          | 0/516 [00:00<?, ?it/s]

In [ ]:
def curate_data(row, audio_dir, lang_extractor, duration_func, audio_func, lang_code):
    audio_path = Path(audio_dir) / row["filename"]
    audio_func(audio_path)
    duration = get_audio_duration(audio_path)
    speaker = "nil"
    lang_per_token = get_lid_tokens(row["transcript"], "yor")
    return {
        "filename": audio_path.stem,
        "audio": str(audio_path),
        "speaker": speaker,
        "transcript": row.get("transcript", "") ,
        "language_id_per_token": lang_per_token,
        "language": lang_code,
        "duration_sec": duration,
    }

def process(df, split, audio_key, audio_dir, log_file, lang_code):
    rows = []
    log_file = f"{log_file}.log"
    processed_log = open(log_file, "a")
    for id, row in tqdm(df.iterrows()):
        if row[audio_key] in {l.strip() for l in open(log_file)} if Path(log_file).exists() else False:
            continue
        try:
            processed = curate_data(row, audio_dir, get_lid_tokens, get_audio_duration, prepare_audio, lang_code)
        except Exception as e:
            print(f"failed row {id} ({row[audio_key]}): {e}")
            continue
        rows.append(processed)
        processed_log.write(row[audio_key] + "\n")
        processed_log.flush()
    processed_log.close()
    return rows

In [ ]:
train_proc = process(train, "train", "filename", DIR  + "flickr_audio_yoruba_train", "train_log", "yor_ng")
val_proc = process(dev, "val", "filename", DIR  + "flickr_audio_yoruba_dev", "val_log", "yor_ng")
test_proc = process(test, "test", "filename", DIR  + "flickr_audio_yoruba_test", "test_log", "yor_ng")
train_proc = Dataset.from_list(train_proc).cast(hf_features)
val_proc = Dataset.from_list(val_proc).cast(hf_features)
test_proc = Dataset.from_list(test_proc).cast(hf_features)
# /content/yor_ng/yfacc_v6/flickr_audio_yoruba_train/S001_1012212859_01547e3f17_0.wav

0it [00:00, ?it/s]

/tmp/ipykernel_53164/839760492.py:27: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sample_rate = librosa.load(file_path, sr=16000)


failed row 70 (S001_1802092493_7b44fdb6b9_0.wav): [Errno 2] No such file or directory: '/content/yor_ng/yfacc_v6/flickr_audio_yoruba_dev/S001_1802092493_7b44fdb6b9_0.wav'
failed row 360 (S001_2987121689_f9de6c479b_0.wav): [Errno 2] No such file or directory: '/content/yor_ng/yfacc_v6/flickr_audio_yoruba_dev/S001_2987121689_f9de6c479b_0.wav'


Casting the dataset:   0%|          | 0/4948 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/514 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/519 [00:00<?, ? examples/s]

In [ ]:
AudDisp(train_proc[0]["audio"]["array"],rate=16_000)

In [ ]:
dataset_dict = DatasetDict({
    "train": train_proc,
    "val" : val_proc,
    "test": test_proc,
})

In [ ]:
dataset_dict.push_to_hub("nolimitsxl/yfacc_yoruba")

Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ? shards/s]

Map:   0%|          | 0/1650 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpavtjmpxk.parquet    :   1%|          |  696kB /  133MB            

CommitInfo(commit_url='https://huggingface.co/datasets/nolimitsxl/yfacc_yoruba/commit/bc5607b24083120b1c528f4c8126eca5d89a1485', commit_message='Upload dataset', commit_description='', oid='bc5607b24083120b1c528f4c8126eca5d89a1485', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nolimitsxl/yfacc_yoruba', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nolimitsxl/yfacc_yoruba'), pr_revision=None, pr_num=None)

### Quick correction

In [ ]:
eng_ng = load_dataset("nolimitsxl/open_slr_lang_resource", "eng_ng")
eng_ng

README.md:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

eng_ng/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  267MB            

eng_ng/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

eng_ng/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  266MB            

eng_ng/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

eng_ng/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  133MB            

eng_ng/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2687 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/672 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['filename', 'audio', 'speaker', 'transcript', 'language_id_per_token', 'language', 'duration_sec'],
        num_rows: 2687
    })
    test: Dataset({
        features: ['filename', 'audio', 'speaker', 'transcript', 'language_id_per_token', 'language', 'duration_sec'],
        num_rows: 672
    })
})

In [ ]:
def fix_lang_id_per_token(batch):
    batch["language_id_per_token"] = ["eng" for _ in batch["language_id_per_token"]]
    return batch

eng_ng = eng_ng.map(fix_lang_id_per_token)


Map:   0%|          | 0/2687 [00:00<?, ? examples/s]

Map:   0%|          | 0/672 [00:00<?, ? examples/s]

In [ ]:
eng_ng.push_to_hub("nolimitsxl/open_slr_lang_resource", config_name="eng_ng",create_pr=True)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/1344 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpcys6_79c.parquet    :  18%|#7        | 23.6MB /  133MB            

CommitInfo(commit_url='https://huggingface.co/datasets/nolimitsxl/open_slr_lang_resource/commit/adc4667129c4f5fe59a70c0b5bbe14975520fff1', commit_message='Upload dataset', commit_description='', oid='adc4667129c4f5fe59a70c0b5bbe14975520fff1', pr_url='https://huggingface.co/datasets/nolimitsxl/open_slr_lang_resource/discussions/1', repo_url=RepoUrl('https://huggingface.co/datasets/nolimitsxl/open_slr_lang_resource', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nolimitsxl/open_slr_lang_resource'), pr_revision='refs/pr/1', pr_num=1)

### DataVerse

In [ ]:
dataverse_files = snapshot_download("nolimitsxl/igbo_sync_raw", revision="main", repo_type="dataset")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
dataverse_files

'/root/.cache/huggingface/hub/datasets--nolimitsxl--igbo_sync_raw/snapshots/37223b62615c211335dbb0bc0fe177c0b4bf98ef'

In [ ]:
ls /root/.cache/huggingface/hub/datasets--nolimitsxl--igbo_sync_raw/snapshots/37223b62615c211335dbb0bc0fe177c0b4bf98ef

 dataverse_files_31-90.zip@     dataverse_files_audio_1-30.zip@   README.md@
'dataverse_files 91_109.zip'@   dataverse_files.zip@


In [ ]:
extract_zip_file(dataverse_files + "/dataverse_files_31-90.zip","/content/igbo_sync_corp")

Successfully extracted '/root/.cache/huggingface/hub/datasets--nolimitsxl--igbo_sync_raw/snapshots/37223b62615c211335dbb0bc0fe177c0b4bf98ef/dataverse_files_31-90.zip' to '/content/igbo_sync_corp'


In [ ]:
extract_zip_file(dataverse_files + "/dataverse_files_audio_1-30.zip","/content/igbo_sync_corp")

Successfully extracted '/root/.cache/huggingface/hub/datasets--nolimitsxl--igbo_sync_raw/snapshots/37223b62615c211335dbb0bc0fe177c0b4bf98ef/dataverse_files_audio_1-30.zip' to '/content/igbo_sync_corp'


In [ ]:
extract_zip_file(dataverse_files + "/dataverse_files 91_109.zip","/content/igbo_sync_corp")

Successfully extracted '/root/.cache/huggingface/hub/datasets--nolimitsxl--igbo_sync_raw/snapshots/37223b62615c211335dbb0bc0fe177c0b4bf98ef/dataverse_files 91_109.zip' to '/content/igbo_sync_corp'


In [ ]:
extract_zip_file(dataverse_files + "/dataverse_files.zip","/content/igbo_sync_corp")

Successfully extracted '/root/.cache/huggingface/hub/datasets--nolimitsxl--igbo_sync_raw/snapshots/37223b62615c211335dbb0bc0fe177c0b4bf98ef/dataverse_files.zip' to '/content/igbo_sync_corp'


In [ ]:
DIR = "/content/igbo_sync_corp"
eaf_files = list(Path(DIR).rglob("*eaf"))

In [ ]:
dev_ids = [1,5,7]
dev_files = [eaf_files[id] for id in dev_ids]
train_files = [eaf_files[id] for id in range(len(eaf_files)) if id not in dev_ids]

In [ ]:
eaf = pympi.Elan.Eaf(train_files[0])

# see what's linked
print(eaf.media_descriptors)  # gives you the audio filename/path it expects

print(eaf.get_tier_names())

for tier_name in eaf.get_tier_names():
    print(eaf.get_annotation_data_for_tier(tier_name))
    for start_ms, end_ms, label in eaf.get_annotation_data_for_tier(tier_name):
        print(tier_name, start_ms, end_ms, label)

In [102]:
def extract_segments(eaf, tier_suffix="tx@", cumsum=25):
    tier = next(t for t in eaf.get_tier_names() if t.startswith(tier_suffix))
    duration, cum_start, cum_end = 0, None, None
    cumsum = cumsum * 1000
    cum_text = ""
    segs = []
    for start, end, text, *_ in eaf.get_annotation_data_for_tier(tier):
        text = text.replace("\n", " ")
        text = re.sub(r"\s*#\s*", " ", text)          # '#' is a pause marker
        text = re.sub(r"\s+", " ", text).strip()
        text = unicodedata.normalize("NFC", text)      # see below
        start = start
        end = end
        duration += end - start
        cum_text += " " + text
        if cum_start is None:
            cum_start, cum_end = start, end
        if duration < cumsum:
            continue
        cum_end = end
        cum_text = cum_text.replace("  ", " ")
        yield cum_start, cum_end, cum_text.strip()
        cum_start, cum_end, cum_text = None, None, ""
        duration = 0
    if cum_text:
        yield cum_start, cum_end, cum_text

In [ ]:
def audio_info(path):
    info = sf.info(path)
    print(f"channels={info.channels} | sr={info.samplerate} Hz | "
          f"duration={info.duration:.1f}s | subtype={info.subtype}")
    return info

info = audio_info("/content/igbo_sync_corp/Anambra_0010.WAV")

channels=2 | sr=48000 Hz | duration=2312.2s | subtype=PCM_16


In [ ]:
def channel_stats(path, block_s=30):
    """Streams the file in blocks, so a 38 min 48kHz stereo file doesn't get loaded at once."""
    with sf.SoundFile(path) as f:
        sr, n_ch = f.samplerate, f.channels
        sumsq = np.zeros(n_ch)
        peak = np.zeros(n_ch)
        cross = 0.0          # sum(L*R), for correlation
        total = 0
        while True:
            d = f.read(int(block_s * sr), dtype="float64", always_2d=True)
            if len(d) == 0:
                break
            sumsq += (d ** 2).sum(axis=0)
            peak = np.maximum(peak, np.abs(d).max(axis=0))
            if n_ch == 2:
                cross += (d[:, 0] * d[:, 1]).sum()
            total += len(d)

    rms = np.sqrt(sumsq / total)
    db = lambda x: 20 * np.log10(np.maximum(x, 1e-12))
    for c in range(n_ch):
        print(f"ch{c}: RMS={db(rms[c]):.1f} dBFS | peak={db(peak[c]):.1f} dBFS")
    if n_ch == 2:
        corr = cross / np.sqrt(sumsq[0] * sumsq[1])
        print(f"level diff = {abs(db(rms[0]) - db(rms[1])):.1f} dB | correlation = {corr:.3f}")
    return rms, (corr if n_ch == 2 else None)

rms, corr = channel_stats("/content/igbo_sync_corp/Abia_0004.wav")

ch0: RMS=-21.2 dBFS | peak=-0.0 dBFS
ch1: RMS=-67.1 dBFS | peak=-57.1 dBFS
level diff = 45.9 dB | correlation = 0.006


In [ ]:
def clip_check(path, ch=0, block_s=30, thresh=0.999):
    clipped = total = 0
    with sf.SoundFile(path) as f:
        sr = f.samplerate
        while True:
            d = f.read(int(block_s * sr), dtype="float32", always_2d=True)
            if len(d) == 0:
                break
            clipped += (np.abs(d[:, ch]) >= thresh).sum()
            total += len(d)
    print(f"clipped samples: {clipped} ({100 * clipped / total:.4f}%)")

clip_check("/content/igbo_sync_corp/Anambra_0010.WAV")

clipped samples: 4225 (0.0038%)


In [123]:
import torch, torchaudio

@lru_cache(maxsize=8)
def _resampler(orig_sr: int, target_sr: int):
    return torchaudio.transforms.Resample(orig_sr, target_sr)

@lru_cache(maxsize=64)
def pick_channel(audio_path: str, block_s=30, dead_db=6.0, dual_mono_corr=0.9):
    """
    Returns (channel, info). channel is an int, or None meaning 'average all'.
    - level diff > dead_db          -> louder channel (dead/weak channel case)
    - similar level, corr > 0.9     -> None (dual mono, averaging is safe)
    - similar level, low/neg corr   -> louder channel + warning (different sources / phase issue)
    """
    with sf.SoundFile(audio_path) as f:
        sr, n_ch = f.samplerate, f.channels
        if n_ch == 1:
            return 0, {"n_ch": 1}
        sumsq = np.zeros(n_ch)
        cross = 0.0
        while True:
            d = f.read(int(block_s * sr), dtype="float64", always_2d=True)
            if len(d) == 0:
                break
            sumsq += (d ** 2).sum(axis=0)
            if n_ch == 2:
                cross += (d[:, 0] * d[:, 1]).sum()

    db = 20 * np.log10(np.sqrt(sumsq / sumsq.sum()) + 1e-12)  # relative levels are all we need
    diff = float(db.max() - db.min())
    corr = float(cross / np.sqrt(sumsq[0] * sumsq[1] + 1e-12)) if n_ch == 2 else None
    loudest = int(np.argmax(sumsq))

    if diff > dead_db:
        ch, why = loudest, "dead_or_weak_channel"
    elif corr is not None and corr > dual_mono_corr:
        ch, why = None, "dual_mono"
    else:
        ch, why = loudest, "ambiguous_check_by_ear"
        print(f"WARNING {Path(audio_path).name}: diff={diff:.1f} dB, corr={corr:.3f}. Listen before trusting.")
    return ch, {"n_ch": n_ch, "level_diff_db": diff, "corr": corr, "reason": why}

def clip_audio(start_ms, end_ms, audio_path, id, out_dir="clips",
               target_sr=16000, channel="auto", clip_thresh=0.999):
    """
    channel: "auto" (per-file pick), an int, or None (average all channels).
    Saves {audio_stem}_{id}.wav (16 kHz mono PCM_16).
    Returns a metadata dict you can append straight to your manifest.
    """
    audio_path = Path(audio_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if channel == "auto":
        channel, ch_info = pick_channel(str(audio_path))
    else:
        ch_info = {"reason": "manual"}

    with sf.SoundFile(audio_path) as f:
        sr = f.samplerate
        start_frame = int(round(start_ms * sr / 1000))
        n_frames = int(round((end_ms - start_ms) * sr / 1000))
        f.seek(start_frame)
        data = f.read(n_frames, dtype="float32", always_2d=True)

    if data.shape[0] == 0:
        raise ValueError(f"Empty clip for id={id}: {start_ms}-{end_ms} ms")

    wav = data.mean(axis=1) if channel is None else data[:, channel]

    # check clipping on the source-rate signal, before resampling smears the peaks
    peak = float(np.abs(wav).max())
    n_clipped = int((np.abs(wav) >= clip_thresh).sum())

    wav_t = torch.from_numpy(wav).unsqueeze(0)
    if sr != target_sr:
        wav_t = _resampler(sr, target_sr)(wav_t)

    out_path = out_dir / f"{audio_path.stem}_{id}.wav"
    sf.write(out_path, wav_t.squeeze(0).numpy(), target_sr, subtype="PCM_16")

    return {
        "audio": str(out_path),
        "source": audio_path.name,
        "filename" : audio_path.stem,
        "id": id,
        "start_ms": start_ms,
        "end_ms": end_ms,
        "duration_sec": (end_ms - start_ms) / 1000,
        "channel": "mean" if channel is None else str(channel),
        "channel_reason": ch_info["reason"],
        "peak": peak,
        "n_clipped": n_clipped,
    }

In [114]:
def get_lid_tokens(transcript):
    return ["ibo" for _ in transcript.split()]

def curate_data(row, lang_extractor, lang_code):
    audio_path = row["file_name"]
    audio_meta = clip_audio(row["start_ms"], row["end_ms"], str(audio_path), row["id"], out_dir="/content/clips")
    speaker = row.get("speaker_id", None)
    lang_per_token = get_lid_tokens(row["transcript"])
    if not isinstance(speaker, str):
        speaker = ""
    return {
        "speaker": speaker,
        "transcript": row.get("transcript", "") ,
        "language_id_per_token": lang_per_token,
        "language": lang_code
    } | audio_meta

In [125]:
def get_audio_path(eaf_path, audio_dir):
    audio_file = "_".join(str(eaf_path.stem).split("_")[:2]) + ".WAV"
    audio_file_2 = "_".join(str(eaf_path.stem).split("_")[:2]) + ".wav"
    if (Path(audio_dir) / audio_file).exists():
        return Path(audio_dir) / audio_file
    elif (Path(audio_dir) / audio_file_2).exists():
        return Path(audio_dir) / audio_file_2
    else:
        raise FileNotFoundError(f"Eaf audio not found{eaf_path}")
def process_dataverse_file(eaf_file, audio_dir, lang_code):
    audio_file = get_audio_path(eaf_file, audio_dir)
    eaf = pympi.Elan.Eaf(eaf_file)
    rows = []
    for id, row in enumerate(extract_segments(eaf)):
        try:
            inst = {
                "file_name": audio_file,
                "id": id,
                "start_ms": row[0],
                "end_ms": row[1],
                "transcript": row[2],
                "speaker_id": "_".join(eaf_file.stem.split("_")[:2])
            }
            if len(inst["transcript"]) <= 1:
                continue
            # print(row[0], row[1])
            processed = curate_data(inst, get_lid_tokens, lang_code)

        except Exception as e:
            print(f"failed row {id} ({row[2]}): {e}")
            continue
        rows.append(processed)
        # break
    return rows

def process(eaf_files, audio_dir, lang_code):
    rows = []
    for row in tqdm(eaf_files):
        processed_file = process_dataverse_file(row, audio_dir, lang_code)
        rows.extend(processed_file)
    return rows

train_data = process(train_files, "/content/igbo_sync_corp", "ibo_ng")
val_data = process(dev_files, "/content/igbo_sync_corp", "ibo_ng")

  0%|          | 0/10 [00:00<?, ?it/s]

WARNING Enugu_0014.wav: diff=4.8 dB, corr=0.318. Listen before trusting.
WARNING Ebonyi_0018.WAV: diff=2.5 dB, corr=0.867. Listen before trusting.


  0%|          | 0/3 [00:00<?, ?it/s]

WARNING Ebonyi_0011.wav: diff=1.3 dB, corr=0.770. Listen before trusting.


In [126]:
train_data[0].keys()

dict_keys(['speaker', 'transcript', 'language_id_per_token', 'language', 'audio', 'source', 'filename', 'id', 'start_ms', 'end_ms', 'duration_sec', 'channel', 'channel_reason', 'peak', 'n_clipped'])

In [127]:
hf_features = Features({
    "filename": Value("string"),
    "audio": Audio(sampling_rate=16000),
    "speaker": Value("string"),
    "transcript": Value("string"),
    "language_id_per_token": List(Value("string")),
    "language": Value("string"),
    "duration_sec": Value("float64"),
    "source": Value("string"),
    "start_ms": Value("float64"),
    "end_ms": Value("float64"),
    "id": Value("int32"),
    "channel": Value("string"),
    "channel_reason": Value("string"),
    "peak": Value("float64"),
    "n_clipped": Value("int32"),
})

In [128]:
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_data).cast(hf_features),
    "val" : Dataset.from_list(val_data).cast(hf_features),
})

Casting the dataset:   0%|          | 0/495 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/175 [00:00<?, ? examples/s]

In [124]:
!rm -rf /content/clips

In [130]:
audio_files = list(Path(DIR).rglob("*.wav")) + list(Path(DIR).rglob("*.WAV"))


In [146]:
def process_dataverse_file_unk(audio_file, audio_dir, lang_code):
    dur = get_audio_duration(audio_file)
    inst = {
        "file_name": audio_file,
        "id": 0,
        "start_ms": 0.0,
        "end_ms": dur * 1000,
        "transcript": "",
        "speaker_id": audio_file.stem
    }
    # print(row[0], row[1])
    processed = curate_data(inst, get_lid_tokens, lang_code)
    return processed

def process(audio_files_files, audio_dir, lang_code):
    rows = []
    for row in tqdm(audio_files_files):
        processed_file = process_dataverse_file_unk(row, audio_dir, lang_code)
        rows.append(processed_file)
    return rows

In [147]:
test_data = process(untranscribed, "/content/igbo_sync_corp", "ibo_ng")

  0%|          | 0/96 [00:00<?, ?it/s]

WARNING Enugu_0008.wav: diff=1.3 dB, corr=0.587. Listen before trusting.
WARNING Enugu_0016.wav: diff=2.7 dB, corr=0.243. Listen before trusting.
WARNING Ebonyi_0008.wav: diff=2.6 dB, corr=0.157. Listen before trusting.
WARNING Enugu_0007.wav: diff=3.5 dB, corr=0.212. Listen before trusting.
WARNING Ebonyi_0012.wav: diff=0.3 dB, corr=0.503. Listen before trusting.
WARNING Enugu_0005.wav: diff=2.8 dB, corr=0.662. Listen before trusting.
WARNING Enugu_0013.wav: diff=5.8 dB, corr=0.897. Listen before trusting.
WARNING Ebonyi_0001.wav: diff=2.1 dB, corr=0.429. Listen before trusting.
WARNING Ebonyi_0003.wav: diff=0.5 dB, corr=0.353. Listen before trusting.
WARNING Enugu_0001.wav: diff=4.5 dB, corr=0.684. Listen before trusting.
WARNING Enugu_0004.wav: diff=0.6 dB, corr=0.497. Listen before trusting.
WARNING Enugu_0003.wav: diff=1.0 dB, corr=0.388. Listen before trusting.
WARNING Enugu_0009.wav: diff=2.5 dB, corr=0.807. Listen before trusting.
WARNING Ebonyi_0010.wav: diff=2.1 dB, corr=0.63

In [148]:
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_data).cast(hf_features),
    "val" : Dataset.from_list(val_data).cast(hf_features),
    "test": Dataset.from_list(test_data).cast(hf_features),
})

Casting the dataset:   0%|          | 0/495 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/175 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

In [149]:
dataset_dict.push_to_hub(
    "nolimitsxl/igbo_sync_processed"
)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/495 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp18nxps4h.parquet    :   0%|          |  575kB /  372MB            

CommitInfo(commit_url='https://huggingface.co/datasets/nolimitsxl/igbo_sync_processed/commit/6bbe4a2027064c1fe3606a45291ce05888e34a72', commit_message='Upload dataset', commit_description='', oid='6bbe4a2027064c1fe3606a45291ce05888e34a72', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nolimitsxl/igbo_sync_processed', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nolimitsxl/igbo_sync_processed'), pr_revision=None, pr_num=None)